# CS 483/583 · Module 16 · Integrated Machine Learning Capstone

Can the class improve on a simple exam-score forecast for students whose scores it has never used? Build a reproducible comparison, find a case where the improvement breaks down, and defend the conclusion in a short report.

Start with Ava, Ben and Cam from Module 01. Add nine fictional records. Reuse tensors from Stage I, regression and model comparison from Stage II, and reliability checks from Module 15. Each row is one student and each score is out of 100.

## Running the examples

Use a Python 3 notebook with PyTorch installed. A CPU is sufficient. Run the cells in order, then change an input and predict what will happen.

Why can getting closer to Ava's score of 60 after a gradient update leave the class unsure about future students?

<details><summary>Reveal answer</summary>

Ava's score helped choose that update. Other students supply independent evidence about whether the changed rule generalizes.

</details>

## Write the question as a prediction task

Put yourself at the instant just before a fictional exam. The model receives a student's study hours and completed practice sets recorded by then. Its target is the exam score, measured later. Success means small errors for students held out from fitting.

For Ava the available pair is (2, 3), and the eventual score is 60. A one-row forecast under the original course rule is 55 points. A post-exam field such as number of correct answers would reveal the outcome.

**Formal notation.** xᵢ = (hᵢ, sᵢ) is student i's pair of hours and sets; yᵢ is that student's exam score. A prediction ŷᵢ has the same unit as yᵢ, score points.

**Predict before you run.** Could you include a field called final_grade when predicting before the exam?

In [1]:
hours, sets, actual = 2, 3, 60
forecast = 20 + 10*hours + 5*sets
print("forecast", forecast, "absolute error", abs(forecast-actual))

forecast 55 absolute error 5


Compare your reasoning with the worked answer.

<details><summary>Reveal answer</summary>

Only information actually available at prediction time belongs in the input. A final grade computed using the exam score leaks the outcome.

</details>

## Freeze the data, split and evaluation rule

Training rows fit parameters. Validation rows help choose a candidate. Test rows are held aside until that choice is fixed. Keep one student in only one split and write down the exact names so someone else can reproduce the same comparison.

Use the six named training records, two validation records and four test records below. Choose among three candidates by lowest validation mean absolute error, resolving a tie in favor of the training mean, then the original rule, then the fitted line. Evaluate the choice once on the test set alongside the predeclared baseline.

**Formal notation.** MAE = (1/n) Σᵢ |ŷᵢ − yᵢ|. Sum absolute errors, then divide by the number n of students in the evaluated split.

**Predict before you run.** After seeing the four test scores, may you try several new models and present the best on those four as an independent result?

In [2]:
# Each row holds a fictional student's name, hours, completed sets, and score.
train = [("Ava",2,3,60),("Ben",4,1,70),("Cam",5,2,75),
         ("Dee",0,1,52),("Eli",1,0,52),("Fin",3,2,66)]
valid = [("Ira",2,0,61),("Jo",4,3,69)]
test = [("Rin",1,1,58),("Sol",3,1,67),("Tay",5,3,68),("Uma",4,0,75)]
names = [name for rows in [train,valid,test] for name,h,s,y in rows]
assert len(names) == len(set(names))
for label, rows in [("train", train), ("validation", valid), ("test", test)]:
    print(label, [row[0] for row in rows])

train ['Ava', 'Ben', 'Cam', 'Dee', 'Eli', 'Fin']
validation ['Ira', 'Jo']
test ['Rin', 'Sol', 'Tay', 'Uma']


Compare your reasoning with the worked answer.

<details><summary>Reveal answer</summary>

Those scores would then help select the model. Treat the change as exploratory and reserve a fresh independent test set for a new evaluation.

</details>

## Set two reference points

A baseline answers how much performance a more complex method must improve upon. Use a constant equal to the mean training score. Also evaluate the original hand-picked two-feature rule so the class can track what it learned since Module 01.

The six training scores sum to 375. Their mean is 62.5, which becomes one shared prediction for every student. On validation, Ira scored 61 and Jo 69. The constant's mean absolute error is (1.5 + 6.5)/2 = 4.

**Formal notation.** c = (Σtrain yᵢ)/ntrain. The constant c uses only training labels. The second candidate is ŷ = 20 + 10h + 5s.

**Predict before you run.** Why should two students with different study histories receive the same baseline forecast?

In [3]:
# Each row holds a fictional student's name, hours, completed sets, and score.
train = [("Ava",2,3,60),("Ben",4,1,70),("Cam",5,2,75),
         ("Dee",0,1,52),("Eli",1,0,52),("Fin",3,2,66)]
valid = [("Ira",2,0,61),("Jo",4,3,69)]
test = [("Rin",1,1,58),("Sol",3,1,67),("Tay",5,3,68),("Uma",4,0,75)]
c = sum(row[3] for row in train)/len(train)
for name,h,s,y in valid:
    print(name, y, c, 20+10*h+5*s)

Ira 61 62.5 40
Jo 69 62.5 75


Compare your reasoning with the worked answer.

<details><summary>Reveal answer</summary>

The baseline deliberately uses only the training mean. Beating it measures whether the features and fitted rule add predictive value on this split.

</details>

## Fit a line using only the training rows

The third candidate uses just study hours. Fit its slope and intercept by minimizing the sum of squared training errors as in Module 05. Evaluate all three candidates using the predeclared MAE. Training and evaluation can use different loss functions when their roles are explicit.

The mean training hours are 2.5 and mean score is 62.5. The sum of centered hour–score products is 87.5; the sum of squared centered hours is 17.5. The fitted slope is 5 and the intercept is 50.

**Formal notation.** w = Σ(hᵢ − h̄)(yᵢ − ȳ)/Σ(hᵢ − h̄)². Set b = ȳ − wh̄. The fitted forecast is b + wh. A bar marks an average over training rows.

**Predict before you run.** What would happen to the slope calculation if every training student reported the same number of hours?

In [4]:
# Each row holds a fictional student's name, hours, completed sets, and score.
train = [("Ava",2,3,60),("Ben",4,1,70),("Cam",5,2,75),
         ("Dee",0,1,52),("Eli",1,0,52),("Fin",3,2,66)]
valid = [("Ira",2,0,61),("Jo",4,3,69)]
test = [("Rin",1,1,58),("Sol",3,1,67),("Tay",5,3,68),("Uma",4,0,75)]
import torch
hours = torch.tensor([h for name,h,sets,y in train], dtype=torch.float64)
scores = torch.tensor([y for name,h,sets,y in train], dtype=torch.float64)
centered = hours - hours.mean()
slope = (centered * (scores-scores.mean())).sum() / centered.square().sum()
bias = scores.mean() - slope * hours.mean()
baseline = scores.mean().item()
def predict(h):
    return bias.item() + slope.item()*h
print("slope, intercept", slope.item(), bias.item())
print("Ava's fitted forecast", predict(2))

slope, intercept 5.0 50.0
Ava's fitted forecast 60.0


Compare your reasoning with the worked answer.

<details><summary>Reveal answer</summary>

Every centered hour value would be zero. The denominator would vanish, so the data could not identify a change in score per extra hour.

</details>

## Select on validation and lock the rule

Evaluate all candidates on the same two validation students, keeping them out of the fit. A difference in validation error informs this example's choice. The small split also limits how stable the ranking may be.

The fitted line predicts 60 for Ira and 70 for Jo, giving errors 1 and 1. The hand-picked rule predicts 40 and 75, giving errors 21 and 6. The constant's errors are 1.5 and 6.5. The fitted line wins the predeclared comparison.

**Formal notation.** m* = argminₘ MAEvalid(m) means choose the candidate m with the smallest validation error. Freeze that choice before inspecting test results.

**Predict before you run.** If the constant had beaten both rules on validation, what should you have selected?

In [5]:
# Each row holds a fictional student's name, hours, completed sets, and score.
train = [("Ava",2,3,60),("Ben",4,1,70),("Cam",5,2,75),
         ("Dee",0,1,52),("Eli",1,0,52),("Fin",3,2,66)]
valid = [("Ira",2,0,61),("Jo",4,3,69)]
test = [("Rin",1,1,58),("Sol",3,1,67),("Tay",5,3,68),("Uma",4,0,75)]
import torch
hours = torch.tensor([h for name,h,sets,y in train], dtype=torch.float64)
scores = torch.tensor([y for name,h,sets,y in train], dtype=torch.float64)
centered = hours - hours.mean()
slope = (centered * (scores-scores.mean())).sum() / centered.square().sum()
bias = scores.mean() - slope * hours.mean()
baseline = scores.mean().item()
def predict(h):
    return bias.item() + slope.item()*h
def mae(rows, prediction):
    errors = [abs(prediction(h,sets)-y) for name,h,sets,y in rows]
    return sum(errors) / len(errors)
models = {"training mean": lambda h,s: baseline,
          "Module 01 rule": lambda h,s: 20+10*h+5*s,
          "fitted hours": lambda h,s: predict(h)}
validation_errors = {name: mae(valid, f) for name,f in models.items()}
print(validation_errors)
print("selected", min(validation_errors, key=validation_errors.get))

{'training mean': 4.0, 'Module 01 rule': 13.5, 'fitted hours': 1.0}
selected fitted hours


Compare your reasoning with the worked answer.

<details><summary>Reveal answer</summary>

Select the constant under the written rule. A richer model must earn its place by predicting held-out outcomes well.

</details>

## Measure the result on the four held-out students

Apply the frozen model and training-mean baseline to all four test students. Show predictions beside outcomes so another student can recompute the reported average. Keep units on the result.

For Rin, Sol, Tay and Uma, the fitted line predicts 55, 65, 75 and 70. Its absolute errors are 3, 2, 7 and 5. The baseline's absolute errors are 4.5, 4.5, 5.5 and 12.5. On these cases the fitted line reduces MAE by 2.5 points.

**Formal notation.** MAEtest(line) = 17/4 = 4.25 points; MAEtest(constant) = 27/4 = 6.75 points. Each mean uses the same four records.

**Predict before you run.** Did the fitted line beat the baseline for every test student?

In [6]:
# Each row holds a fictional student's name, hours, completed sets, and score.
train = [("Ava",2,3,60),("Ben",4,1,70),("Cam",5,2,75),
         ("Dee",0,1,52),("Eli",1,0,52),("Fin",3,2,66)]
valid = [("Ira",2,0,61),("Jo",4,3,69)]
test = [("Rin",1,1,58),("Sol",3,1,67),("Tay",5,3,68),("Uma",4,0,75)]
import torch
hours = torch.tensor([h for name,h,sets,y in train], dtype=torch.float64)
scores = torch.tensor([y for name,h,sets,y in train], dtype=torch.float64)
centered = hours - hours.mean()
slope = (centered * (scores-scores.mean())).sum() / centered.square().sum()
bias = scores.mean() - slope * hours.mean()
baseline = scores.mean().item()
def predict(h):
    return bias.item() + slope.item()*h
def mae(rows, prediction):
    errors = [abs(prediction(h,sets)-y) for name,h,sets,y in rows]
    return sum(errors) / len(errors)
models = {"training mean": lambda h,s: baseline,
          "Module 01 rule": lambda h,s: 20+10*h+5*s,
          "fitted hours": lambda h,s: predict(h)}
for name,h,s,y in test:
    print(name, "observed", y, "line", predict(h), "baseline", baseline)
print("test MAE", mae(test, models["fitted hours"]), mae(test, models["training mean"]))

Rin observed 58 line 55.0 baseline 62.5
Sol observed 67 line 65.0 baseline 62.5
Tay observed 68 line 75.0 baseline 62.5
Uma observed 75 line 70.0 baseline 62.5
test MAE 4.25 6.75


Compare your reasoning with the worked answer.

<details><summary>Reveal answer</summary>

Tay's line error is 7 versus the baseline's 5.5. An improvement in the average can coexist with an individual regression.

</details>

## Interpret a probability only under its assumptions

A point prediction does not describe the spread around it. For an illustration, assume scores around the line have a Gaussian shape with a fixed standard deviation of five points. This specified spread is an additional modeling assumption. Evaluate its calibration with additional data before interpreting it as a measured uncertainty.

For four hours the mean is 70. A symmetric Gaussian centered at 70 puts half its mass at or above 70. For three hours the mean is 65; reaching 70 is one assumed standard deviation above it, an upper tail near 0.159.

**Formal notation.** Y | h ∼ Normal(50 + 5h, 5²) describes a conditional score distribution. The second argument 5² is the variance. P(Y ≥ 70 | h) is the fraction of that distribution at or above 70.

**Predict before you run.** Does 0.5 at four hours mean that half of future four-hour students have been shown to reach 70?

In [7]:
import torch
for h in [3., 4.]:
    mu = torch.tensor(50 + 5*h)
    distribution = torch.distributions.Normal(mu, torch.tensor(5.))
    upper_tail = 1 - distribution.cdf(torch.tensor(70.))
    print(h, round(upper_tail.item(), 3))

3.0 0.159
4.0 0.5


Compare your reasoning with the worked answer.

<details><summary>Reveal answer</summary>

That fraction follows from the assumed Gaussian curve. Check observed frequencies in a larger held-out sample and evaluate the assumed spread.

</details>

## Report a failure and test the recording assumption

Separate prediction errors by a rule stated in the report and show the counts. Also simulate a plausible input mistake while holding the original scores fixed. These exploratory checks identify what to investigate on fresh data.

For students with at least four hours, the line's errors are 7 and 5, averaging 6. For the two students below four hours they average 2.5. If a reporting bug adds one hour to each test record, each line prediction rises by five points. Recalculate all four errors rather than assuming each gets worse.

**Formal notation.** Residual eᵢ = ŷᵢ − yᵢ. Positive means overprediction. The paired stress test uses h′ᵢ = hᵢ + 1 and keeps yᵢ fixed.

**Predict before you run.** Can these two groups establish that studying longer causes larger prediction errors?

In [8]:
# Each row holds a fictional student's name, hours, completed sets, and score.
train = [("Ava",2,3,60),("Ben",4,1,70),("Cam",5,2,75),
         ("Dee",0,1,52),("Eli",1,0,52),("Fin",3,2,66)]
valid = [("Ira",2,0,61),("Jo",4,3,69)]
test = [("Rin",1,1,58),("Sol",3,1,67),("Tay",5,3,68),("Uma",4,0,75)]
import torch
hours = torch.tensor([h for name,h,sets,y in train], dtype=torch.float64)
scores = torch.tensor([y for name,h,sets,y in train], dtype=torch.float64)
centered = hours - hours.mean()
slope = (centered * (scores-scores.mean())).sum() / centered.square().sum()
bias = scores.mean() - slope * hours.mean()
baseline = scores.mean().item()
def predict(h):
    return bias.item() + slope.item()*h
for minimum, maximum in [(0, 3), (4, 5)]:
    errors = [abs(predict(h)-y) for name,h,s,y in test if minimum <= h <= maximum]
    print(minimum, maximum, len(errors), sum(errors)/len(errors))
shifted = [abs(predict(h+1)-y) for name,h,s,y in test]
print("shifted errors", shifted, "MAE", sum(shifted)/len(shifted))

0 3 2 2.5
4 5 2 6.0
shifted errors [2.0, 3.0, 12.0, 0.0] MAE 4.25


Compare your reasoning with the worked answer.

<details><summary>Reveal answer</summary>

Each group contains only two students. Study behavior was observed, and the groups may differ in other ways. Report the observed errors and test the pattern on fresh records.

</details>

## Submit a claim someone else can check

Submit a runnable notebook and a one-page findings report. Include the prediction moment, feature definitions, original rows and split, training code, baseline, selection rule, test table, and one stress test. Conclude with an error you would investigate and the evidence you would need.

A defensible conclusion is that the training-fitted hours line reduced test MAE from 6.75 to 4.25 on these four fictional students; it missed Tay by seven points. A fresh cohort and independently verified study logs could test whether the improvement persists. Award one rubric point for each reproducible element: task and split, fitting and selection, predictions and metrics, and a supported limitation with a follow-up test.

**Formal notation.** Improvement = MAEbaseline − MAEselected = 2.50 score points on this split. A positive difference favors the selected model under this metric.

**Predict before you run.** A reviewer gets exactly your numbers but concludes that one more hour of study will raise anyone's score by five points. What should you revise?

In [9]:
# Each row holds a fictional student's name, hours, completed sets, and score.
train = [("Ava",2,3,60),("Ben",4,1,70),("Cam",5,2,75),
         ("Dee",0,1,52),("Eli",1,0,52),("Fin",3,2,66)]
valid = [("Ira",2,0,61),("Jo",4,3,69)]
test = [("Rin",1,1,58),("Sol",3,1,67),("Tay",5,3,68),("Uma",4,0,75)]
import torch
hours = torch.tensor([h for name,h,sets,y in train], dtype=torch.float64)
scores = torch.tensor([y for name,h,sets,y in train], dtype=torch.float64)
centered = hours - hours.mean()
slope = (centered * (scores-scores.mean())).sum() / centered.square().sum()
bias = scores.mean() - slope * hours.mean()
baseline = scores.mean().item()
def predict(h):
    return bias.item() + slope.item()*h
def mae(rows, prediction):
    errors = [abs(prediction(h,sets)-y) for name,h,sets,y in rows]
    return sum(errors) / len(errors)
models = {"training mean": lambda h,s: baseline,
          "Module 01 rule": lambda h,s: 20+10*h+5*s,
          "fitted hours": lambda h,s: predict(h)}
winner = min(models, key=lambda name: mae(valid, models[name]))
base_error = mae(test, models["training mean"])
final_error = mae(test, models[winner])
print(winner, "baseline MAE", base_error, "model MAE", final_error)
print("observed improvement", base_error-final_error)

fitted hours baseline MAE 6.75 model MAE 4.25
observed improvement 2.5


Compare your reasoning with the worked answer.

<details><summary>Reveal answer</summary>

Make the interpretation explicit: five is the fitted slope on fictional training records. A causal effect would require evidence about what would happen if study hours actually changed.

</details>

## Exercise 1

Use a novel student with 2.5 hours and 1 set. Predict using the frozen line, training mean and Module 01 rule. If the score is 64, which is closest?

<details><summary>Hint</summary>

Use the already fitted coefficients without modifying any model.

</details>

Check your work.

<details><summary>Reveal answer</summary>

The line and constant both predict 62.5, each 1.5 points away. The original rule predicts 20 + 25 + 5 = 50, 14 points away.

</details>

## Exercise 2

Build and defend a comparison for an in-class variant with new named records.

<details><summary>Hint</summary>

Write the prediction moment, split and choice rule before fitting.

</details>

Check your work.

<details><summary>Reveal answer</summary>

Choose a pre-exam target, keep each student in one split, fit the constant and a feature-based model on training only, select by validation MAE and evaluate the frozen choice and baseline on common test records. Report split names, predictions and errors so another student can reproduce the comparison.

</details>

## Exercise 3

A new blind test contains scores 63 at two hours and 72 at five hours. Compare the frozen line and constant.

<details><summary>Hint</summary>

Use 50 + 5h and the training mean of 62.5.

</details>

Check your work.

<details><summary>Reveal answer</summary>

Line predictions 60 and 75 give errors 3 and 3, MAE 3. Constant errors 0.5 and 9.5 give MAE 5. This small independent check favors the line by 2 points.

</details>

## Exercise 4

A team uses the test scores to select a different standard deviation for each student and reports excellent likelihood on those same scores. Identify the problem and repair the design.

<details><summary>Hint</summary>

Which records chose the uncertainty parameters?

</details>

Check your work.

<details><summary>Reveal answer</summary>

The test labels influenced the predictions being scored. Fit a spread on training or calibration data, freeze it, and evaluate the same likelihood or calibration criterion on unseen students.

</details>

## Exercise 5

For Tay and Uma compute signed residuals. What does their mean hide?

<details><summary>Hint</summary>

Subtract the observed score from the fitted prediction for each student.

</details>

Check your work.

<details><summary>Reveal answer</summary>

Tay gives 75 − 68 = +7. Uma gives 70 − 75 = −5. Their signed mean is +1, but their mean absolute error is 6. The mistakes point in opposite directions.

</details>

## Exercise 6

Propose one improvement after seeing the four test rows and describe how to test it fairly.

<details><summary>Hint</summary>

Separate discovering an idea from confirming its performance.

</details>

Check your work.

<details><summary>Reveal answer</summary>

One proposal is to fit a two-feature model using both hours and completed sets. Fit on training, compare on validation, and use a new untouched test cohort because the existing test set helped motivate the change. Report errors for both the new model and the original frozen line.

</details>

Return to the course roadmap with a complete argument: a prediction rule, a reproducible fit, an independent comparison, and an honest account of where the evidence runs out. In a new setting, begin by writing down one decision, one observable target and one test that could change your conclusion.

Which evidence would most directly strengthen the claim that the fitted line helps on future exams?

<details><summary>Reveal answer</summary>

Collect a larger independent cohort with the same prediction-time features, apply the frozen line and baseline, and compare errors on the same new students while checking performance across relevant conditions.

</details>